In [ ]:
! pip install youtube_transcript_api langchain-community langchain

In [ ]:
! pip install langchain-huggingface huggingface-hub sentence-transformers torch accelerate

In [ ]:
! pip install langchain-google-genai>=4.0.0

In [ ]:
! pip install langchain-chroma>=1.0.0 chromadb>=1.0.0

In [ ]:
! pip install langchain_core

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace,HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from dotenv import load_dotenv

In [6]:
load_dotenv()

False

# Indexing

In [13]:
video_id = "Gfr50f6ZBvo"

## Document Ingestion

In [14]:
try:
    api = YouTubeTranscriptApi()

    # Ask YouTube which transcripts are available.
    available_transcripts = list(api.list(video_id))

    if not available_transcripts:
        raise RuntimeError("No transcripts are available for this video.")

    # Prefer English if available; otherwise use the first available language.
    transcript_to_fetch = next(
        (
            transcript
            for transcript in available_transcripts
            if transcript.language_code in {"en", "en-US"}
        ),
        available_transcripts[0],
    )

    transcript = transcript_to_fetch.fetch()

    document = Document(
        page_content=" ".join(
            snippet.text for snippet in transcript.snippets
        ),
        metadata={
            "source": f"https://www.youtube.com/watch?v={video_id}",
            "language": transcript_to_fetch.language_code,
            "generated": transcript_to_fetch.is_generated,
        },
    )

    print("Selected language:", document.metadata["language"])
    print("Auto-generated:", document.metadata["generated"])
    print()
    print(document.page_content[:1000])

except Exception as error:
    print(f"Transcript unavailable: {error}")

Selected language: en
Auto-generated: True

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote 

In [15]:
document.page_content[:500]

'the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful '

## Text Splitting

In [16]:
splilter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, chunk_overlap = 200
)
chunks = splilter.create_documents([document.page_content])

In [17]:
len(chunks)

168

In [18]:
chunks[8]

Document(metadata={}, page_content="thing that truly impresses other humans so that you can have a close friend who's an ai system for that friend to be a good friend they're going to have to be able to uh play starcraft and they're going to have to do all of these tasks um get you a beer so the robotics tasks uh play games with you use language humor all those kinds of things but that ultimately can boil down to language it feels like not in terms of the ai community but in terms of the actual impact of general intelligence on the world it feels like language will be the place where it truly shines i think so because it's such an important kind of input output for us i think you're right i think the turing test that what the the kind of the the philosophy behind it which is the idea of can can a machine mimic the behaviors of a human and men and i would say wider than just language and text then you know in terms of actions and everything else creativity all these things then um if it

## Embedding Generation and Storing in Vector Store

In [19]:
import os
from google.colab import userdata

# Fallback to manual text input if the secret is not shared or not configured
try:
    google_key = userdata.get("GEMINI_API_KEY")
    os.environ["GOOGLE_API_KEY"] = google_key
except Exception:
    import getpass
    print("Please provide your Gemini API key directly (or make sure you toggled Notebook Access in Colab Secrets):")
    google_key = getpass.getpass("Enter GEMINI_API_KEY: ")
    os.environ["GEMINI_API_KEY"] = google_key

gemini_embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-2')
hf_embedding = HuggingFaceEmbeddings(model_name= 'sentence-transformers/all-MiniLM-L6-v2')
vector_store = Chroma(
    embedding_function=hf_embedding,
    persist_directory="./chroma_db",
    collection_name='youtube_chat'
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
os.environ['HF_HOME'] = "/workspaces/gen-ai/hf_models"

In [21]:
doc_ids = vector_store.add_documents(documents=chunks)

In [22]:
len(doc_ids)

168

In [23]:
doc_ids

['10664ffb-8d34-4a1f-8319-92c5533fc495',
 '06ee0e9a-1025-4dd8-83f5-4c7c6b96796f',
 'c93d6fbc-989f-4d79-b42f-2354624f7edc',
 '1d0fb011-518e-4544-8300-52977b1c5ac0',
 'ed3a5d6a-4266-4057-b8b2-ec3a2facc064',
 '7e4189d1-d8af-447b-89bd-2e608667c9a0',
 '75de6123-a743-4be0-a12e-209c00b26857',
 '60b3e42a-bcaa-409d-9b34-69d052a8559c',
 '065f1db5-d813-4056-a9e3-72caf2605314',
 '7d885413-5d82-4d42-80e8-661435e8848f',
 '24bc29ff-b094-439e-a649-5d8c80fded41',
 '87d408a0-c4fc-4838-9477-3e97bc16691e',
 '8ac6446b-422e-4300-b5fe-882d7c6f4630',
 'f040408a-9e0d-4ae9-97a0-35215dca8a5e',
 '17d66fa9-af02-43e4-842f-9b1761f0a5d9',
 '401fed2f-83ee-4400-ac9c-d51bc7aad471',
 'c7920e99-5542-47a7-88c9-647cae7c81fe',
 'a90c217c-e276-4ed9-9a79-7e58b2a4b944',
 '2f3206ae-10ea-4d73-8c60-4d173945f033',
 'c9aab9a3-2a05-4526-98a1-1b4c5b5076e4',
 'd297199e-2e1e-4ca0-b75c-859d658edf3e',
 '7b102ade-7566-4a0e-a8aa-f0a44b479060',
 '9e964f97-1320-4d47-9ac9-5a5ce3cbc644',
 '8ed697c6-73fa-41dc-8702-77afd905fa05',
 '9820b3a6-daad-

In [26]:
# Fetch raw results directly to avoid the LangChain-Chroma validation bug
raw_results = vector_store.get(ids=['552a377f-f307-442c-9535-a5afbb7600ea'], include=['embeddings', 'documents', 'metadatas'])

# # Safely rebuild the Document objects by supplying empty dictionaries if metadata is None
# retrieved_docs = []
# for doc, meta, doc_id in zip(raw_results["documents"], raw_results["metadatas"] or [None]*len(raw_results["documents"]), raw_results["ids"]):
#     retrieved_docs.append(Document(page_content=doc, metadata=meta or {}, id=doc_id))

# retrieved_docs

In [27]:
raw_results['documents'][0]

"deeper maybe simpler explanation yes of things right than the standard model of physics which we know doesn't work but we still keep adding to so um and and that's how i think the beginning of an explanation would look and it would start encompassing many of the mysteries that we have wondered about for thousands of years like you know consciousness uh life and gravity all of these things yeah giving us a glimpses of explanations for those things yeah well um damas dear one of the special human beings in this giant puzzle of ours and it's a huge honor that you would take a pause from the bigger puzzle to solve this small puzzle of a conversation with me today it's truly an honor and a pleasure thank you thank you i really enjoyed it thanks lex thanks for listening to this conversation with demas establish to support this podcast please check out our sponsors in the description and now let me leave you with some words from edskar dykstra computer science is no more about computers than

In [28]:
vector_store.get(['552a377f-f307-442c-9535-a5afbb7600ea'], include=['embeddings', 'documents'])

{'ids': ['552a377f-f307-442c-9535-a5afbb7600ea'],
 'embeddings': array([[-1.36370152e-01, -6.41418202e-03,  6.12084158e-02,
          3.97891738e-03,  6.66581616e-02, -5.83931543e-02,
          1.56879760e-02, -9.24895145e-03,  4.01258022e-02,
          1.44612845e-02, -4.82179299e-02, -1.78258971e-03,
         -1.00894779e-01, -4.73071896e-02, -2.04086490e-02,
         -8.94610304e-03,  2.11526677e-02, -8.77224132e-02,
         -1.50621012e-01,  6.34047315e-02,  4.56983447e-02,
         -1.92553774e-02,  1.29668124e-03,  2.32696645e-02,
         -5.13309538e-02,  4.30860557e-02,  1.52963698e-02,
         -2.62629222e-02,  2.20652577e-02, -8.39238912e-02,
          3.76323462e-02,  4.14697602e-02,  6.46909624e-02,
          4.32190932e-02, -2.31845118e-02,  1.00279815e-01,
          2.38309801e-02, -6.59477711e-02, -3.13981473e-02,
         -6.34561479e-02, -4.93148156e-03, -6.26926310e-03,
          3.81366052e-02,  1.03501879e-01,  7.94008933e-03,
         -7.46322703e-03, -5.7971701

# Retrieval

In [29]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [30]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7b5bc01181a0>, search_kwargs={'k': 4})

In [31]:
retriever.invoke('What is RTX 6000 ?')

[Document(id='1fe9f2e1-06d1-4c25-a0e6-96b4c97aa48b', metadata={}, page_content='demas establish to support this podcast please check out our sponsors in the description and now let me leave you with some words from edskar dykstra computer science is no more about computers than astronomy is about telescopes thank you for listening and hope to see you next time'),
 Document(id='46401944-0ff7-4fb4-8ff6-82538100a87c', metadata={}, page_content="behind it in other ways some other projects will do non-profit style um and also we have to consider for future things as well safety and ethics as well like but you know synthetic biology there are you know there is dual use and we have to think about that as well with alpha fold we you know we consulted with 30 different bioethicists and and other people expert in this field to make sure it was safe before um we released it so there'll be other considerations in future but for right now you know i think alpha fold is a kind of a gift from us to t

# Augmentation

In [32]:
gemini_llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.2)

hf_llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen3-0.6B",
    task= "text-generation",
    pipeline_kwargs=dict(
        temperature = 0.5,
        max_new_tokens = 50
    )
)
hf_model = ChatHuggingFace(llm=hf_llm)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [33]:
prompt = PromptTemplate(
    template= """
    You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables=["context", "question"]
)

In [34]:
question  = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)


In [35]:
retrieved_docs

[Document(id='311f0986-cb86-436f-b7b2-0c345aa50a26', metadata={}, page_content="in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be intere

In [36]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be interesting from a research perspective from our point of view from an ai point of\n\

In [37]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

## Generation

In [ ]:
answer = hf_model.invoke(final_prompt)
answer.content

In [39]:
# Use Gemini to generate a clean answer since the local hf_model echoed the prompt template
gemini_answer = gemini_llm.invoke(final_prompt)
print(gemini_answer.content)

Yes, the topic of nuclear fusion is discussed in the video.

What was discussed:
*   Nuclear fusion is identified as one of the biggest areas where AI can help, alongside disease and biology.
*   It presents many challenges, including physics, material science, and engineering, particularly in building massive fusion reactors and containing plasma.
*   The approach taken is to collaborate with domain experts, identify bottleneck problems, and determine which of these are amenable to AI methods.
*   They collaborated with EPFL in Switzerland, using their test reactor for experiments.
*   They published a Nature paper titled "magnetic control of tokamak plasmas to deep reinforcement learning."
*   This work involved using AI (specifically deep reinforcement learning) to control high-temperature plasmas.
*   They successfully developed a controller that could hold plasma in specific shapes and contain it for a record amount of time, even carving it into different shapes (like "droplets" w

# Chaining

In [42]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [43]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [44]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [45]:
parallel_chain.invoke('Tell me about demis')

{'context': "to get world peace because there's also other corrupting things like wanting power over people and this kind of stuff which is not necessarily satisfied by by just abundance but i think it will help um and i think uh but i think ultimately ai is not going to be run by any one person or one organization i think it should belong to the world belong to humanity um and i think maybe many there'll be many ways this will happen and ultimately um everybody should have a say in that do you have advice for uh young people in high school and college maybe um if they're interested in ai or interested in having a big impact on the world what they should do to have a career they can be proud of her to have a life they can be proud of i love giving talks to the next generation what i say to them is actually two things i i think the most important things to learn about and to find out about when you're when you're young is what are your true passions is first of all there's two things on